# Pairwise Ranking Router Training

This notebook trains a VLM router using **pairwise ranking** approach.

## Approach

Instead of predicting absolute rewards, we learn to **rank models** by comparing pairs:
- For each (sample, mode), create pairs (model_i, model_j) where model_i outperforms model_j
- Train router to assign higher scores to better models
- Loss: Margin Ranking Loss - encourage score(model_i) > score(model_j) + margin

## Benefits

- **Relative judgments**: Easier than absolute reward prediction
- **Robust to calibration**: Only ranking matters, not absolute scores
- **Proven in ranking tasks**: Used in learning-to-rank, recommendation systems

## Sections

1. Setup & Configuration
2. Load Data from SQL
3. Compute Rewards & Generate Pairs
4. Build Pairwise Dataset
5. Train Pairwise Router
6. Evaluate & Compare to Oracle
7. Visualizations

## 1. Setup & Configuration

In [ ]:
# Add parent directory to path
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import logging
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Local imports
from config import DBConfig, RewardWeights
from db_utils import load_profiles_real_schema, test_connection
from reward_definitions import compute_rewards_real_schema
from training.pairwise_dataset import (
    generate_pairwise_examples,
    PairwiseRouterDataset,
    collate_pairwise_batch,
)
from models.pairwise_router import PairwiseRouterModel

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Configuration
LIMIT = 1000  # Set to None for full dataset, or use small number for testing
DATA_SPLIT = "train"  # Load only training data

# Pairwise generation settings
MIN_MARGIN = 0.05  # Minimum reward difference to include pair
MAX_PAIRS_PER_SAMPLE = 20  # Limit pairs per (sample, mode) to avoid explosion

# Training settings
BATCH_SIZE = 16
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
MARGIN = 1.0  # Margin for ranking loss
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Paths
DATA_DIR = Path("../data")
MODELS_DIR = Path("../models/checkpoints")
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)

print("Configuration:")
print(f"  LIMIT: {LIMIT}")
print(f"  DATA_SPLIT: {DATA_SPLIT}")
print(f"  MIN_MARGIN: {MIN_MARGIN}")
print(f"  MAX_PAIRS_PER_SAMPLE: {MAX_PAIRS_PER_SAMPLE}")
print(f"  BATCH_SIZE: {BATCH_SIZE}")
print(f"  NUM_EPOCHS: {NUM_EPOCHS}")
print(f"  LEARNING_RATE: {LEARNING_RATE}")
print(f"  MARGIN: {MARGIN}")
print(f"  DEVICE: {DEVICE}")

## 2. Load Data from SQL

In [ ]:
# Load database configuration
db_config = DBConfig.from_env()

# Test connection
print("Testing database connection...")
if test_connection(db_config):
    print("✓ Database connection successful")
else:
    raise RuntimeError("Database connection failed. Check credentials.")

In [ ]:
# Load profiling data from SQL
print(f"\nLoading profiling data (data_split={DATA_SPLIT}, limit={LIMIT})...")

df_profiles = load_profiles_real_schema(
    db_config=db_config,
    limit=LIMIT,
    data_split=DATA_SPLIT,
)

print(f"\n✓ Loaded {len(df_profiles)} profile records")
print(f"  Unique samples: {df_profiles['sample_id'].nunique()}")
print(f"  Unique models: {df_profiles['model_name'].nunique()}")
print(f"  Models: {sorted(df_profiles['model_name'].unique())}")

# Show sample
df_profiles.head()

## 3. Compute Rewards & Generate Pairs

In [ ]:
# Compute multi-objective rewards
print("\nComputing rewards...")

reward_weights = RewardWeights()  # Use defaults
df_with_rewards = compute_rewards_real_schema(df_profiles, reward_weights)

print(f"✓ Computed rewards for {len(df_with_rewards)} records")
print("\nReward columns:")
reward_cols = [c for c in df_with_rewards.columns if c.startswith('reward_')]
print(reward_cols)

# Show reward statistics
print("\nReward statistics:")
print(df_with_rewards[reward_cols].describe())

In [ ]:
# Expand to include all 4 modes
print("\nExpanding dataset to include all 4 routing modes...")

modes = ['accuracy', 'cheap', 'fast', 'balanced']
expanded_rows = []

for _, row in df_with_rewards.iterrows():
    for mode in modes:
        row_dict = row.to_dict()
        row_dict['mode_id'] = mode
        row_dict['reward'] = row[f'reward_{mode}']
        expanded_rows.append(row_dict)

df_expanded = pd.DataFrame(expanded_rows)

print(f"✓ Expanded to {len(df_expanded)} rows ({len(df_with_rewards)} → {len(df_expanded)})")
print(f"  Rows per mode: {len(df_expanded) // 4}")

df_expanded.head()

In [ ]:
# Generate pairwise examples for training
print(f"\nGenerating pairwise examples (metric=reward, min_margin={MIN_MARGIN})...")

df_pairs_train = generate_pairwise_examples(
    df=df_expanded,
    metric_column="reward",
    min_margin=MIN_MARGIN,
    max_pairs_per_sample=MAX_PAIRS_PER_SAMPLE,
)

print(f"\n✓ Generated {len(df_pairs_train)} pairwise examples")
print(f"  From {df_pairs_train['sample_id'].nunique()} unique samples")
print(f"  Pairs per sample: {len(df_pairs_train) / df_pairs_train['sample_id'].nunique():.1f} avg")

# Show pair statistics
print("\nPair margin statistics:")
print(df_pairs_train['margin'].describe())

df_pairs_train.head()

In [ ]:
# Visualize pair margin distribution
plt.figure(figsize=(10, 5))
plt.hist(df_pairs_train['margin'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Margin (reward_i - reward_j)')
plt.ylabel('Count')
plt.title('Distribution of Pairwise Margins')
plt.axvline(MIN_MARGIN, color='red', linestyle='--', label=f'Min margin = {MIN_MARGIN}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Median margin: {df_pairs_train['margin'].median():.3f}")
print(f"Mean margin: {df_pairs_train['margin'].mean():.3f}")

## 4. Build Pairwise Dataset

In [ ]:
# Create model and mode mappings
unique_models = sorted(df_expanded['model_name'].unique())
unique_modes = sorted(df_expanded['mode_id'].unique())

model_to_id = {model: i for i, model in enumerate(unique_models)}
id_to_model = {i: model for model, i in model_to_id.items()}

mode_to_id = {mode: i for i, mode in enumerate(unique_modes)}
id_to_mode = {i: mode for mode, i in mode_to_id.items()}

print(f"Models ({len(model_to_id)}): {model_to_id}")
print(f"Modes ({len(mode_to_id)}): {mode_to_id}")

# Save mappings
with open(DATA_DIR / "model_index_pairwise.json", "w") as f:
    json.dump(model_to_id, f, indent=2)
with open(DATA_DIR / "mode_index_pairwise.json", "w") as f:
    json.dump(mode_to_id, f, indent=2)

print("\n✓ Saved model and mode index mappings")

In [ ]:
# Create PyTorch dataset
train_dataset = PairwiseRouterDataset(
    pairs_df=df_pairs_train,
    model_to_id=model_to_id,
    mode_to_id=mode_to_id,
)

print(f"✓ Created training dataset: {len(train_dataset)} examples")

# Create dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_pairwise_batch,
)

print(f"✓ Created training dataloader: {len(train_loader)} batches")

# Test batch
sample_batch = next(iter(train_loader))
print("\nSample batch keys:", sample_batch.keys())
print("  sample_texts:", len(sample_batch['sample_texts']))
print("  model_i_ids:", sample_batch['model_i_ids'].shape)
print("  model_j_ids:", sample_batch['model_j_ids'].shape)
print("  mode_ids:", sample_batch['mode_ids'].shape)
print("  labels:", sample_batch['labels'].shape)

## 5. Train Pairwise Router

In [ ]:
# Initialize model
model = PairwiseRouterModel(
    num_models=len(model_to_id),
    num_modes=len(mode_to_id),
    text_encoder_name="distilbert-base-uncased",
    model_embed_dim=32,
    mode_embed_dim=16,
    hidden_dim=256,
    dropout=0.1,
)

model = model.to(DEVICE)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print(f"\n✓ Initialized model on {DEVICE}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Training loop
print(f"\nTraining for {NUM_EPOCHS} epochs...\n")

history = {
    'train_loss': [],
    'pairwise_accuracy': [],  # % of pairs where score_i > score_j
}

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch in pbar:
        optimizer.zero_grad()
        
        # Forward pass
        loss = model.compute_pairwise_loss(
            sample_texts=batch['sample_texts'],
            model_i_ids=batch['model_i_ids'],
            model_j_ids=batch['model_j_ids'],
            mode_ids=batch['mode_ids'],
            margin=MARGIN,
        )
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        epoch_loss += loss.item()
        
        # Compute pairwise accuracy (score_i > score_j)
        with torch.no_grad():
            scores_i = model(
                batch['sample_texts'],
                batch['model_i_ids'],
                batch['mode_ids'],
            )
            scores_j = model(
                batch['sample_texts'],
                batch['model_j_ids'],
                batch['mode_ids'],
            )
            correct = (scores_i > scores_j).sum().item()
            epoch_correct += correct
            epoch_total += len(scores_i)
        
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'acc': f"{100 * correct / len(scores_i):.1f}%"
        })
    
    # Epoch statistics
    avg_loss = epoch_loss / len(train_loader)
    pairwise_acc = 100 * epoch_correct / epoch_total
    
    history['train_loss'].append(avg_loss)
    history['pairwise_accuracy'].append(pairwise_acc)
    
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Pairwise Acc = {pairwise_acc:.2f}%")

print("\n✓ Training complete")

In [ ]:
# Save trained model
model_path = MODELS_DIR / "best_pairwise_router.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'model_to_id': model_to_id,
    'mode_to_id': mode_to_id,
    'config': {
        'num_models': len(model_to_id),
        'num_modes': len(mode_to_id),
        'text_encoder_name': 'distilbert-base-uncased',
        'model_embed_dim': 32,
        'mode_embed_dim': 16,
        'hidden_dim': 256,
        'dropout': 0.1,
    },
}, model_path)

print(f"✓ Saved model to {model_path}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], marker='o', label='Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Pairwise accuracy
axes[1].plot(history['pairwise_accuracy'], marker='o', color='green', label='Pairwise Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Pairwise Ranking Accuracy')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Evaluate & Compare to Oracle

In [ ]:
# Load validation data
print("Loading validation data...")

df_val_profiles = load_profiles_real_schema(
    db_config=db_config,
    limit=500,  # Smaller validation set
    data_split="val",
)

# Compute rewards
df_val_with_rewards = compute_rewards_real_schema(df_val_profiles, reward_weights)

# Expand modes
val_expanded_rows = []
for _, row in df_val_with_rewards.iterrows():
    for mode in modes:
        row_dict = row.to_dict()
        row_dict['mode_id'] = mode
        row_dict['reward'] = row[f'reward_{mode}']
        val_expanded_rows.append(row_dict)

df_val_expanded = pd.DataFrame(val_expanded_rows)

print(f"✓ Loaded {len(df_val_expanded)} validation rows")
print(f"  Unique samples: {df_val_expanded['sample_id'].nunique()}")

In [ ]:
# Evaluate routing accuracy vs oracle
print("\nEvaluating routing accuracy...\n")

model.eval()
results = []

# Group by (sample_id, mode_id)
grouped = df_val_expanded.groupby(['sample_id', 'mode_id'])

for (sample_id, mode_id), group in tqdm(grouped, desc="Evaluating"):
    # Oracle choice (best reward)
    oracle_choice = group.loc[group['reward'].idxmax(), 'model_name']
    oracle_reward = group['reward'].max()
    
    # Router prediction
    sample_text = group.iloc[0]['prompt_raw']  # Same for all models
    available_models = group['model_name'].tolist()
    available_model_ids = [model_to_id[m] for m in available_models]
    
    with torch.no_grad():
        predicted_ids = model.predict_best_model(
            sample_texts=[sample_text],
            model_ids=available_model_ids,
            mode_id=mode_to_id[mode_id],
        )
        predicted_model_id = predicted_ids[0]
        predicted_model = id_to_model[predicted_model_id]
    
    # Get router's reward
    router_reward = group.loc[group['model_name'] == predicted_model, 'reward'].values[0]
    
    results.append({
        'sample_id': sample_id,
        'mode_id': mode_id,
        'oracle_choice': oracle_choice,
        'oracle_reward': oracle_reward,
        'router_choice': predicted_model,
        'router_reward': router_reward,
        'correct': oracle_choice == predicted_model,
        'reward_gap': oracle_reward - router_reward,
    })

df_results = pd.DataFrame(results)

# Print results
overall_acc = 100 * df_results['correct'].mean()
avg_gap = df_results['reward_gap'].mean()
median_gap = df_results['reward_gap'].median()

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"Routing Accuracy: {overall_acc:.2f}%")
print(f"Average Reward Gap: {avg_gap:.4f}")
print(f"Median Reward Gap: {median_gap:.4f}")
print(f"Oracle Reward (avg): {df_results['oracle_reward'].mean():.4f}")
print(f"Router Reward (avg): {df_results['router_reward'].mean():.4f}")
print("="*60)

# Per-mode breakdown
print("\nPer-mode breakdown:")
for mode in modes:
    mode_results = df_results[df_results['mode_id'] == mode]
    mode_acc = 100 * mode_results['correct'].mean()
    mode_gap = mode_results['reward_gap'].mean()
    print(f"  {mode:12s}: Acc = {mode_acc:5.2f}%, Gap = {mode_gap:.4f}")

## 7. Visualizations

In [ ]:
# Routing accuracy by mode
mode_acc = df_results.groupby('mode_id')['correct'].mean() * 100

plt.figure(figsize=(10, 5))
plt.bar(mode_acc.index, mode_acc.values, color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Mode')
plt.ylabel('Routing Accuracy (%)')
plt.title('Routing Accuracy by Mode')
plt.axhline(overall_acc, color='red', linestyle='--', label=f'Overall = {overall_acc:.2f}%')
plt.ylim([0, 100])
plt.grid(True, alpha=0.3, axis='y')
plt.legend()
plt.show()

In [ ]:
# Reward gap distribution
plt.figure(figsize=(10, 5))
plt.hist(df_results['reward_gap'], bins=50, edgecolor='black', alpha=0.7, color='coral')
plt.xlabel('Reward Gap (Oracle - Router)')
plt.ylabel('Count')
plt.title('Distribution of Reward Gap')
plt.axvline(0, color='green', linestyle='--', linewidth=2, label='Perfect routing (gap=0)')
plt.axvline(avg_gap, color='red', linestyle='--', label=f'Mean gap = {avg_gap:.4f}')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

perfect_routing = (df_results['reward_gap'] == 0).mean() * 100
print(f"\nPerfect routing rate: {perfect_routing:.2f}%")

In [ ]:
# Oracle vs Router rewards scatter
plt.figure(figsize=(8, 8))
plt.scatter(
    df_results['oracle_reward'],
    df_results['router_reward'],
    alpha=0.5,
    s=30,
    c=df_results['correct'].map({True: 'green', False: 'red'}),
)
plt.plot([0, 1], [0, 1], 'k--', label='Perfect routing')
plt.xlabel('Oracle Reward')
plt.ylabel('Router Reward')
plt.title('Oracle vs Router Rewards')
plt.grid(True, alpha=0.3)
plt.legend(['Perfect routing', 'Correct choice', 'Incorrect choice'])
plt.axis('equal')
plt.show()

In [ ]:
# Confusion matrix of model choices
from sklearn.metrics import confusion_matrix
import itertools

# Get oracle and router choices as model IDs
oracle_ids = [model_to_id[m] for m in df_results['oracle_choice']]
router_ids = [model_to_id[m] for m in df_results['router_choice']]

cm = confusion_matrix(oracle_ids, router_ids)

plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix: Oracle vs Router Choices')
plt.colorbar()
tick_marks = np.arange(len(model_to_id))
plt.xticks(tick_marks, [id_to_model[i] for i in range(len(model_to_id))], rotation=45, ha='right')
plt.yticks(tick_marks, [id_to_model[i] for i in range(len(model_to_id))])

# Add text annotations
thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, format(cm[i, j], 'd'),
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")

plt.ylabel('Oracle Choice')
plt.xlabel('Router Choice')
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated training a **pairwise ranking router** for VLM routing:

### Key Results
- **Pairwise Training Accuracy**: Measures how well the model ranks model pairs
- **Routing Accuracy**: How often the router picks the same model as the oracle
- **Reward Gap**: Difference between oracle and router rewards

### Next Steps
1. **Tune hyperparameters**: Adjust margin, learning rate, min_margin
2. **Compare approaches**: Run `04_classical_ce_kl_router.ipynb` to compare
3. **Test set evaluation**: Evaluate on held-out test set
4. **Production deployment**: Use trained model for real-time routing

### Files Generated
- `../models/checkpoints/best_pairwise_router.pt` - Trained model
- `../data/model_index_pairwise.json` - Model ID mappings
- `../data/mode_index_pairwise.json` - Mode ID mappings